### Modelling for Q2

Import the dataset used for training the XGBoost Model

In [1]:
import pandas as pd


In [2]:
data = pd.read_csv("../data/curated/data_XGBoost_before_onehotencode.csv")

In [6]:
data = pd.get_dummies(data, columns=["property type", "closest_city_centre"])

In [40]:
y_train = data["weekly rent amount"]
X_train = data.drop(["weekly rent amount"], axis=1)

Train model

In [41]:
import xgboost as xgb
from sklearn.metrics import r2_score
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=70, learning_rate=0.1, enable_categorical=True)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=70, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

Now create the test dataset by incorporating the predicted census data for 2026

In [12]:
test_data = pd.read_csv("../data/curated/data_XGBoost_before_onehotencode_with_postcode.csv")
# test_data = pd.get_dummies(test_data, columns=["property type", "closest_city_centre"])

In [13]:
test_data.head()

,property type,beds,baths,parking,weekly rent amount,postcode,distance primary school,distance secondary school,is closest school good,distance university campus,...,closest_city_centre,coastal,closest_cc_dist,dist_to_melb,predicted median equivalised total household income (weekly) ($) 2023,predicted median weekly household rental payment ($) 2023,predicted average family size (no. of persons) 2023,predicted employment ratio 2023,predicted estimated resident population (no.) 2023,predicted total dwellings (no.) 2023
0,House,1,2,6,120,3127,0.586504,2.491226,True,7.430317,...,Melbourne,False,11.566742,12.618789,1567.422222,487.077778,3.030000,0.952619,42748.566667,18091.000000
1,House,9,2,2,130,3216,1.372612,2.438191,False,0.675843,...,Geelong,False,7.459818,71.172437,1172.308333,403.983333,3.019167,0.956643,76343.466667,31411.666667
2,House,1,1,0,140,3936,4.858273,3.301447,True,47.278807,...,Cranbourne,False,37.174742,58.248165,985.433333,429.133333,2.700000,0.967222,13759.066667,9143.000000
3,House,1,2,6,140,3216,1.114181,1.998214,False,1.502495,...,Geelong,False,7.459818,71.172437,1172.308333,403.983333,3.019167,0.956643,76343.466667,31411.666667
4,House,1,4,3,150,3216,0.215009,1.989564,False,1.862509,...,Geelong,False,7.459818,71.172437,1172.308333,403.983333,3.019167,0.956643,76343.466667,31411.666667


In [14]:
postcode_info_2026 = pd.read_csv("../data/curated/postcode_info_2026.csv")

In [16]:
postcode_info_2026["Employment Ratio"] = postcode_info_2026["Employed (no.)"] / (postcode_info_2026["Employed (no.)"] + postcode_info_2026["Unemployed (no.)"])

In [17]:
postcode_info_2026.head()

,Postcode,Estimated resident population (no.),Employed (no.),Unemployed (no.),In the Labour force (no.),Total persons aged 15 years and over (no.),Land area (ha),Total families (no.),Total dwellings (no.),Median equivalised total household income (weekly) ($),Median weekly household rental payment ($),Average family size (no. of persons),Employment Ratio
0,3000,157031.0,119719.0,10414.0,129242.0,180530.0,0.0,41988.0,108362.0,1589.865710,423.457739,2.510361,0.919974
1,3002,4718.0,3057.0,97.0,3151.0,4599.0,0.0,1260.0,3671.0,2172.310652,513.350649,2.302083,0.969245
2,3003,24107.0,18438.0,1307.0,19641.0,25913.0,0.0,6567.0,16037.0,1513.938021,384.182262,2.501000,0.933806
3,3004,109191.0,91135.0,7559.0,98106.0,132932.0,0.0,29229.0,83269.0,1595.423448,411.422507,2.407273,0.923410
4,3006,40600.0,42974.0,2582.0,45964.0,61478.0,0.0,15497.0,32288.0,1823.873490,456.478563,2.504422,0.943323


In [19]:
restrict_post_info = postcode_info_2026[["Postcode", "Median equivalised total household income (weekly) ($)", 
                                         "Median weekly household rental payment ($)", 
                                         "Average family size (no. of persons)", "Employment Ratio",
                                         "Estimated resident population (no.)", "Total dwellings (no.)"]]


Merge the datasets based on the Postcode

In [20]:
test_data_merged = test_data.merge(restrict_post_info, left_on='postcode', right_on='Postcode', how='inner')

In [21]:
test_data_merged.head()

,property type,beds,baths,parking,weekly rent amount,postcode,distance primary school,distance secondary school,is closest school good,distance university campus,...,predicted employment ratio 2023,predicted estimated resident population (no.) 2023,predicted total dwellings (no.) 2023,Postcode,Median equivalised total household income (weekly) ($),Median weekly household rental payment ($),Average family size (no. of persons),Employment Ratio,Estimated resident population (no.),Total dwellings (no.)
0,House,1,2,6,120,3127,0.586504,2.491226,True,7.430317,...,0.952619,42748.566667,18091.0,3127,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
1,Apartment / Unit / Flat,1,1,0,420,3127,0.951010,0.846483,False,9.427377,...,0.952619,42748.566667,18091.0,3127,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
2,Apartment / Unit / Flat,2,1,0,480,3127,0.449996,1.917181,True,10.270417,...,0.952619,42748.566667,18091.0,3127,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
3,Apartment / Unit / Flat,2,1,1,495,3127,0.824231,1.077620,False,9.001152,...,0.952619,42748.566667,18091.0,3127,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
4,Apartment / Unit / Flat,3,1,1,580,3127,0.232646,2.008062,False,8.589805,...,0.952619,42748.566667,18091.0,3127,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0


In [23]:
test_data.columns

Index(['property type', 'beds', 'baths', 'parking', 'weekly rent amount',
       'postcode', 'distance primary school', 'distance secondary school',
       'is closest school good', 'distance university campus',
       'distance to closest hospital', 'offence count 2023',
       'distance train station', 'city_centre', 'closest_city_centre',
       'coastal', 'closest_cc_dist', 'dist_to_melb',
       'predicted median equivalised total household income (weekly) ($) 2023',
       'predicted median weekly household rental payment ($) 2023',
       'predicted average family size (no. of persons) 2023',
       'predicted employment ratio 2023',
       'predicted estimated resident population (no.) 2023',
       'predicted total dwellings (no.) 2023'],
      dtype='object')

Drop the unnecessary columns

In [47]:
final_test_data = test_data_merged.drop(["postcode", 'predicted median equivalised total household income (weekly) ($) 2023',
       'predicted median weekly household rental payment ($) 2023',
       'predicted average family size (no. of persons) 2023',
       'predicted employment ratio 2023',
       'predicted estimated resident population (no.) 2023',
       'predicted total dwellings (no.) 2023', "Postcode", "weekly rent amount"], axis = 1)

In [33]:
data.columns

Index(['beds', 'baths', 'parking', 'weekly rent amount',
       'distance primary school', 'distance secondary school',
       'is closest school good', 'distance university campus',
       'distance to closest hospital', 'offence count 2023',
       'distance train station', 'city_centre', 'coastal', 'closest_cc_dist',
       'dist_to_melb',
       'predicted median equivalised total household income (weekly) ($) 2023',
       'predicted median weekly household rental payment ($) 2023',
       'predicted average family size (no. of persons) 2023',
       'predicted employment ratio 2023',
       'predicted estimated resident population (no.) 2023',
       'predicted total dwellings (no.) 2023',
       'property type_Acreage / Semi-Rural',
       'property type_Apartment / Unit / Flat', 'property type_Duplex',
       'property type_House', 'property type_New Apartments / Off the Plan',
       'property type_New House & Land', 'property type_Penthouse',
       'property type_Retirement'

In [48]:
final_test_data.columns

Index(['property type', 'beds', 'baths', 'parking', 'distance primary school',
       'distance secondary school', 'is closest school good',
       'distance university campus', 'distance to closest hospital',
       'offence count 2023', 'distance train station', 'city_centre',
       'closest_city_centre', 'coastal', 'closest_cc_dist', 'dist_to_melb',
       'Median equivalised total household income (weekly) ($)',
       'Median weekly household rental payment ($)',
       'Average family size (no. of persons)', 'Employment Ratio',
       'Estimated resident population (no.)', 'Total dwellings (no.)'],
      dtype='object')

Change the naming of the columns to fit the XGBoost Model columns

In [49]:
column_mapping = {
    'Median equivalised total household income (weekly) ($)': 'predicted median equivalised total household income (weekly) ($) 2023',
    'Median weekly household rental payment ($)': 'predicted median weekly household rental payment ($) 2023',
    'Average family size (no. of persons)': 'predicted average family size (no. of persons) 2023',
    'Employment Ratio': 'predicted employment ratio 2023',
    'Estimated resident population (no.)': 'predicted estimated resident population (no.) 2023',
    'Total dwellings (no.)': 'predicted total dwellings (no.) 2023'
}

final_test_data.rename(columns=column_mapping, inplace=True)


In [50]:
final_test_data.head()

,property type,beds,baths,parking,distance primary school,distance secondary school,is closest school good,distance university campus,distance to closest hospital,offence count 2023,...,closest_city_centre,coastal,closest_cc_dist,dist_to_melb,predicted median equivalised total household income (weekly) ($) 2023,predicted median weekly household rental payment ($) 2023,predicted average family size (no. of persons) 2023,predicted employment ratio 2023,predicted estimated resident population (no.) 2023,predicted total dwellings (no.) 2023
0,House,1,2,6,0.586504,2.491226,True,7.430317,2.104719,29.0,...,Melbourne,False,11.566742,12.618789,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
1,Apartment / Unit / Flat,1,1,0,0.951010,0.846483,False,9.427377,0.551691,29.0,...,Melbourne,False,11.566742,12.618789,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
2,Apartment / Unit / Flat,2,1,0,0.449996,1.917181,True,10.270417,2.142153,29.0,...,Melbourne,False,11.566742,12.618789,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
3,Apartment / Unit / Flat,2,1,1,0.824231,1.077620,False,9.001152,2.431453,29.0,...,Melbourne,False,11.566742,12.618789,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0
4,Apartment / Unit / Flat,3,1,1,0.232646,2.008062,False,8.589805,1.702026,29.0,...,Melbourne,False,11.566742,12.618789,1752.858167,525.240369,2.984929,0.954928,41786.0,18088.0


One hot Encode the Categorical Variables

In [51]:
final_test_data = pd.get_dummies(final_test_data, columns=["property type", "closest_city_centre"])

### Predictions

In [52]:
predictions = model.predict(final_test_data)

In [53]:
print(predictions)

[ 485.0993   425.43277  545.7984  ...  646.5926  1473.6772  1777.6538 ]


In [54]:
final_test_data.head()

,beds,baths,parking,distance primary school,distance secondary school,is closest school good,distance university campus,distance to closest hospital,offence count 2023,distance train station,...,closest_city_centre_Horsham,closest_city_centre_Melbourne,closest_city_centre_Morwell,closest_city_centre_Sale,closest_city_centre_Shepparton,closest_city_centre_Swan Hill,closest_city_centre_Traralgon,closest_city_centre_Wangaratta,closest_city_centre_Warrnambool,closest_city_centre_Wodonga
0,1,2,6,0.586504,2.491226,True,7.430317,2.104719,29.0,1.667194,...,0,1,0,0,0,0,0,0,0,0
1,1,1,0,0.951010,0.846483,False,9.427377,0.551691,29.0,0.672151,...,0,1,0,0,0,0,0,0,0,0
2,2,1,0,0.449996,1.917181,True,10.270417,2.142153,29.0,1.148754,...,0,1,0,0,0,0,0,0,0,0
3,2,1,1,0.824231,1.077620,False,9.001152,2.431453,29.0,0.236749,...,0,1,0,0,0,0,0,0,0,0
4,3,1,1,0.232646,2.008062,False,8.589805,1.702026,29.0,0.467375,...,0,1,0,0,0,0,0,0,0,0


In [56]:
predictions[-10:]

array([ 908.56433,  840.7181 ,  903.19476,  834.761  , 1469.3262 ,
        861.3406 , 1146.3271 ,  646.5926 , 1473.6772 , 1777.6538 ],
      dtype=float32)

Add predictions as a column and save the dataset

In [57]:
final_test_data["predicted weekly rent amount"] = predictions

In [58]:
final_test_data.to_csv("../data/curated/predictions_2026.csv", index=False)

### Look at the growth rate between 2023 and 2026 data

In [60]:
postcodes = pd.read_csv("../data/curated/data_XGBoost_before_onehotencode_with_postcode.csv")

postcodes = postcodes["postcode"]

In [61]:
final_test_data["postcode"] = postcodes

In [71]:
agg_data_2026 = final_test_data.groupby('postcode')['predicted weekly rent amount'].mean().reset_index()

agg_data_2026 = agg_data_2026.rename(columns={"predicted weekly rent amount": "weekly rent amount 2026"})

In [72]:
data_2023 = pd.read_csv("../data/curated/data_XGBoost_before_onehotencode_with_postcode.csv")

agg_data_2023 = data_2023.groupby('postcode')['weekly rent amount'].mean().reset_index()

agg_data_2023 = agg_data_2023.rename(columns={"weekly rent amount": "weekly rent amount 2023"})

In [73]:
agg_data = agg_data_2023.merge(agg_data_2026, on="postcode", how="inner")

In [74]:
agg_data.head()

,postcode,weekly rent amount 2023,weekly rent amount 2026
0,3000,671.313536,673.147156
1,3002,809.151515,704.014954
2,3003,600.779070,684.126770
3,3004,709.277108,705.886963
4,3006,713.786667,646.616882


In [76]:
agg_data["growth_rate"] = agg_data["weekly rent amount 2026"] / agg_data["weekly rent amount 2023"] - 1

In [81]:
agg_data.sort_values(by=['growth_rate'], ascending=False)

,postcode,weekly rent amount 2023,weekly rent amount 2026,growth_rate
278,3638,180.0,763.226196,3.240146
211,3318,240.0,871.392822,2.630803
252,3477,250.0,889.919556,2.559678
410,3988,390.0,1134.429565,1.908794
206,3289,230.0,624.856079,1.716766
...,...,...,...,...
74,3091,1047.5,616.658386,-0.411305
350,3835,1477.0,826.730652,-0.440264
226,3357,850.0,433.050446,-0.490529
90,3115,1850.0,795.159485,-0.570184


In [82]:
agg_data.to_csv("../data/curated/weekly_rent_2023_and_2026.csv", index=False)